# 02 Evaluation Report

Experiment report notebook for the Myanmar G2P PBSMT assignment.

## Purpose

This notebook collects Moses EMS evaluation outputs and existing CSV artifacts, summarizes root-cause analysis with Myanmar G2P terminology, and writes a report-style `README.md`.

No score values are hardcoded. The notebook first loads `pbsmt_experiment_results.csv` if it exists. If the CSV is missing, it parses the Moses evaluation folders under `pbsmt_original_clean_experiments`.

In [1]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import re

BASE_DIR = Path('/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein')
G2P_DIR = BASE_DIR / 'g2p-par'
EXP_DIR = BASE_DIR / 'pbsmt_original_clean_experiments'

print('BASE_DIR:', BASE_DIR)
print('G2P_DIR:', G2P_DIR)
print('EXP_DIR:', EXP_DIR)

BASE_DIR: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein
G2P_DIR: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par
EXP_DIR: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments


## Load score and error-analysis CSVs

In [2]:
def read_text(p):
    return p.read_text(encoding='utf-8', errors='ignore') if p.exists() else ''


def clean_out(line):
    return ' '.join(re.sub(r'\|\d+-\d+\|', '', line).split()).strip()


def parse_report(txt):
    # Moses EMS report format: test: 75.90 (1.000) BLEU-c ; 75.90 (1.000) BLEU
    m = re.search(r'test:\s*([0-9.]+)\s*\(([0-9.]+)\)\s*BLEU-c\s*;\s*([0-9.]+)\s*\(([0-9.]+)\)\s*BLEU', txt)
    if m:
        return {
            'bleu_c': float(m.group(1)),
            'nist_c': float(m.group(2)),
            'bleu': float(m.group(3)),
            'nist': float(m.group(4)),
        }
    return {}


def exact_stats(eval_dir):
    f_out = eval_dir / 'test.output.1'
    f_ref = eval_dir / 'test.reference.txt.1'
    if not (f_out.exists() and f_ref.exists()):
        return {}
    total = errors = 0
    for out, ref in zip(f_out.open(encoding='utf-8'), f_ref.open(encoding='utf-8')):
        total += 1
        if clean_out(out) != ' '.join(ref.split()).strip():
            errors += 1
    return {
        'test_items': total,
        'exact_match_errors': errors,
        'exact_match_accuracy': round((total - errors) / total * 100, 4) if total else None,
    }


def collect_results_from_experiment_dirs():
    rows = []
    for eval_dir in sorted(EXP_DIR.glob('*/*/evaluation')):
        row = {
            'experiment': eval_dir.parts[-3],
            'direction': eval_dir.parts[-2],
            'evaluation_dir': str(eval_dir),
            'report_exists': (eval_dir / 'report.1').exists(),
        }
        row.update(parse_report(read_text(eval_dir / 'report.1')))
        row.update(exact_stats(eval_dir))
        rows.append(row)
    return pd.DataFrame(rows)


def load_results():
    """Load existing score CSV first. If missing, parse Moses experiment folders.

    No score values are hardcoded in this notebook.
    """
    candidates = [
        Path('pbsmt_experiment_results.csv'),
        BASE_DIR / 'pbsmt_experiment_results.csv',
        EXP_DIR / 'pbsmt_experiment_results.csv',
    ]
    for p in candidates:
        if p.exists():
            print('Loading score CSV:', p)
            return pd.read_csv(p)

    print('No score CSV found. Parsing Moses evaluation folders from:', EXP_DIR)
    df = collect_results_from_experiment_dirs()
    if not df.empty:
        df.to_csv('pbsmt_experiment_results.csv', index=False)
        print('Wrote parsed score CSV: pbsmt_experiment_results.csv')
    return df


def load_error_summary():
    """Load error summary CSV generated by error-analysis notebook.

    Expected columns can be either:
    - Root_Cause, Count, Percentage
    - root_cause, count, percentage
    """
    candidates = [
        Path('pbsmt_error_summary.csv'),
        Path('error_summary_pbsmt_5gram_main_my-ph.csv'),
        Path('error_summary_pbsmt_7gram_phrase11_my-ph.csv'),
        *sorted(Path('.').glob('error_summary_*.csv')),
        *sorted(Path('.').glob('final_error_summary_*.csv')),
    ]
    for p in candidates:
        if p.exists():
            print('Loading error summary CSV:', p)
            df = pd.read_csv(p)
            df.columns = [c.strip() for c in df.columns]
            return df
    print('No error summary CSV found. Run 03_error_analysis.ipynb first if you want the error-analysis table in README.md.')
    return pd.DataFrame()


def load_leaderboard():
    candidates = [
        Path('error_leaderboard_pbsmt_5gram_main_my-ph.csv'),
        Path('error_leaderboard_pbsmt_7gram_phrase11_my-ph.csv'),
        *sorted(Path('.').glob('error_leaderboard_*.csv')),
        *sorted(Path('.').glob('final_error_leaderboard_*.csv')),
    ]
    for p in candidates:
        if p.exists():
            print('Loading error leaderboard CSV:', p)
            df = pd.read_csv(p)
            df.columns = [c.strip() for c in df.columns]
            return df
    print('No error leaderboard CSV found.')
    return pd.DataFrame()


def normalize_error_labels(df):
    """Rename analysis labels for report wording.

    Previous code used 'Train Data Ambiguity (Both Exist)'. For Myanmar G2P, this is better described as
    'Context-dependent pronunciation variation' because a syllable/word may naturally have different pronunciations
    depending on lexical item and phonological context.
    """
    if df.empty:
        return df
    out = df.copy()
    cause_col = 'Root_Cause' if 'Root_Cause' in out.columns else ('root_cause' if 'root_cause' in out.columns else None)
    if cause_col:
        out[cause_col] = out[cause_col].replace({
            'Train Data Ambiguity (Both Exist)': 'Context-dependent pronunciation variation',
            'Reference Missing from Training Token': 'Missing pronunciation variant in training data',
            'Decoder Hallucination / Over-generalization': 'Rare over-generalization / segmentation error',
        })
    return out


results_df = load_results()
if not results_df.empty:
    # normalize columns expected from older notebooks
    results_df.columns = [c.strip() for c in results_df.columns]
    if 'exact_match_errors' in results_df.columns:
        sort_cols = ['direction', 'exact_match_errors']
        results_df = results_df.sort_values(sort_cols).reset_index(drop=True)
    results_df.to_csv('pbsmt_experiment_results.csv', index=False)

error_summary_df = normalize_error_labels(load_error_summary())
leaderboard_df = normalize_error_labels(load_leaderboard())

if not error_summary_df.empty:
    error_summary_df.to_csv('pbsmt_error_summary_report_labels.csv', index=False)

if not leaderboard_df.empty:
    leaderboard_df.to_csv('pbsmt_error_leaderboard_report_labels.csv', index=False)

results_df

Loading score CSV: pbsmt_experiment_results.csv
Loading error summary CSV: pbsmt_error_summary.csv
Loading error leaderboard CSV: error_leaderboard_pbsmt_5gram_main_my-ph.csv


,experiment,direction,evaluation_dir,report_exists,bleu_c,nist_c,bleu,nist,test_items,exact_match_errors,exact_match_accuracy
0,pbsmt_7gram_phrase11,my-ph,/media/kyalkalay/Data/AI-Class/assignment-6_au...,True,75.80,1.001,75.80,1.001,2802,998,64.3826
1,pbsmt_3gram_phrase7,my-ph,/media/kyalkalay/Data/AI-Class/assignment-6_au...,True,75.93,0.999,75.93,0.999,2802,1000,64.3112
2,pbsmt_5gram_main,my-ph,/media/kyalkalay/Data/AI-Class/assignment-6_au...,True,75.90,1.000,75.90,1.000,2802,1002,64.2398
3,pbsmt_3gram_phrase5,my-ph,/media/kyalkalay/Data/AI-Class/assignment-6_au...,True,75.89,0.999,75.89,0.999,2802,1002,64.2398
4,pbsmt_5gram_phrase13,my-ph,/media/kyalkalay/Data/AI-Class/assignment-6_au...,True,75.72,1.001,75.72,1.001,2802,1005,64.1328
5,pbsmt_5gram_phrase7,my-ph,/media/kyalkalay/Data/AI-Class/assignment-6_au...,True,75.75,1.000,75.75,1.000,2802,1007,64.0614
6,pbsmt_5gram_nbest200,my-ph,/media/kyalkalay/Data/AI-Class/assignment-6_au...,True,75.69,1.000,75.69,1.000,2802,1007,64.0614
7,pbsmt_5gram_search10000,my-ph,/media/kyalkalay/Data/AI-Class/assignment-6_au...,True,75.74,1.000,75.74,1.000,2802,1009,63.9900
8,pbsmt_7gram_phrase9,my-ph,/media/kyalkalay/Data/AI-Class/assignment-6_au...,True,75.73,0.999,75.73,0.999,2802,1010,63.9543
9,pbsmt_5gram_phrase11,my-ph,/media/kyalkalay/Data/AI-Class/assignment-6_au...,True,75.57,1.000,75.57,1.000,2802,1012,63.8829


## Write README.md report

In [3]:
def pct_text(value):
    try:
        return f"{float(value):.2f}%"
    except Exception:
        return str(value)


def get_col(df, names):
    for n in names:
        if n in df.columns:
            return n
    return None


result_section = '_No score results were found. Run 01_pbsmt_training.ipynb and this notebook again._'
comparison_section = '_No comparison could be generated because score results were not found._'

if not results_df.empty:
    show_cols = [c for c in ['experiment', 'direction', 'bleu_c', 'nist_c', 'bleu', 'nist', 'test_items', 'exact_match_errors', 'exact_match_accuracy'] if c in results_df.columns]
    myph = results_df[results_df['direction'] == 'my-ph'].copy() if 'direction' in results_df.columns else pd.DataFrame()
    phmy = results_df[results_df['direction'] == 'ph-my'].copy() if 'direction' in results_df.columns else pd.DataFrame()

    result_section = ''
    if not myph.empty:
        result_section += '### my-ph Direction\n\n'
        result_section += myph[show_cols].to_markdown(index=False) + '\n\n'
    if not phmy.empty:
        result_section += '### ph-my Direction\n\n'
        result_section += phmy[show_cols].to_markdown(index=False) + '\n\n'

    comparison_lines = []
    for direction, group in [('my-ph', myph), ('ph-my', phmy)]:
        if group.empty or 'exact_match_errors' not in group.columns:
            continue
        best = group.sort_values(['exact_match_errors', 'bleu_c'] if 'bleu_c' in group.columns else ['exact_match_errors'], ascending=[True, False] if 'bleu_c' in group.columns else [True]).iloc[0]
        baseline_rows = group[group['experiment'] == 'pbsmt_5gram_main'] if 'experiment' in group.columns else pd.DataFrame()
        if not baseline_rows.empty:
            baseline = baseline_rows.iloc[0]
            gain_errors = int(baseline['exact_match_errors'] - best['exact_match_errors'])
            gain_acc = float(best['exact_match_accuracy'] - baseline['exact_match_accuracy']) if 'exact_match_accuracy' in group.columns else 0.0
            comparison_lines.append(
                f"- **{direction}**: best system is `{best['experiment']}` with {int(best['exact_match_errors'])} exact-match errors. "
                f"Compared with `pbsmt_5gram_main`, this changes the result by {gain_errors} errors and {gain_acc:.4f} accuracy points."
            )
        else:
            comparison_lines.append(
                f"- **{direction}**: best system is `{best['experiment']}` with {int(best['exact_match_errors'])} exact-match errors."
            )
    comparison_section = '\n'.join(comparison_lines) if comparison_lines else comparison_section

error_section = '_No error-analysis summary was found. Run 03_error_analysis.ipynb and this notebook again._'
error_interpretation = ''
if not error_summary_df.empty:
    error_section = error_summary_df.to_markdown(index=False)
    cause_col = get_col(error_summary_df, ['Root_Cause', 'root_cause'])
    pct_col = get_col(error_summary_df, ['Percentage', 'percentage'])
    count_col = get_col(error_summary_df, ['Count', 'count'])

    if cause_col:
        # Find context-dependent row after relabeling
        ctx = error_summary_df[error_summary_df[cause_col].astype(str).str.contains('Context-dependent pronunciation', regex=False)]
        oov = error_summary_df[error_summary_df[cause_col].astype(str).str.contains('Out of Vocabulary', regex=False)]
        missing = error_summary_df[error_summary_df[cause_col].astype(str).str.contains('Missing pronunciation variant', regex=False)]

        if not ctx.empty:
            ctx_row = ctx.iloc[0]
            ctx_pct = ctx_row[pct_col] if pct_col else ''
            ctx_count = ctx_row[count_col] if count_col else ''
            error_interpretation += (
                f"The largest group is **context-dependent pronunciation variation** "
                f"({ctx_count} cases, {ctx_pct}%). This is not simply a training-data mistake: "
                f"it reflects the nature of Myanmar G2P, where some graphemes or words have different pronunciations depending on lexical item, neighboring syllables, and phonological context.\n\n"
            )

        data_issue_parts = []
        if not oov.empty:
            data_issue_parts.append(f"OOV: {int(oov.iloc[0][count_col]) if count_col else 'N/A'}")
        if not missing.empty:
            data_issue_parts.append(f"missing pronunciation variant: {int(missing.iloc[0][count_col]) if count_col else 'N/A'}")
        if data_issue_parts:
            error_interpretation += (
                "The remaining substantial errors are data coverage/preparation issues "
                f"({', '.join(data_issue_parts)}). These should be handled through data fixing, dictionary expansion, and verified pronunciation-variant additions.\n"
            )

leaderboard_section = ''
if not leaderboard_df.empty:
    cause_col = get_col(leaderboard_df, ['Root_Cause', 'root_cause'])
    if cause_col:
        ref_missing = leaderboard_df[leaderboard_df[cause_col].astype(str).str.contains('Missing pronunciation variant', regex=False)].copy()
        if not ref_missing.empty:
            keep_cols = [c for c in ['Source_Syllable', 'Moses_Output', 'True_Reference', 'Root_Cause', 'Error_Count'] if c in ref_missing.columns]
            leaderboard_section = '\n\n### Examples of Missing Pronunciation Variants\n\n' + ref_missing[keep_cols].head(30).to_markdown(index=False)

readme = f"""# Myanmar G2P PBSMT Experiment Report

## Abstract

This experiment evaluates Phrase-Based Statistical Machine Translation (PBSMT) for Myanmar grapheme-to-phoneme conversion using the myG2P dataset. The confirmed baseline is a 5-gram Moses PBSMT system generated from the original assignment template. Additional 3-gram, 5-gram tuning, and 7-gram systems were tested to determine whether n-gram order, phrase length, pruning, MERT n-best size, or stronger search improves exact-match accuracy.

The tuning experiments produced only marginal changes. This indicates that the main remaining limitation is not PBSMT search or simple hyperparameter choice. The error analysis points instead to Myanmar G2P pronunciation variation and data coverage/preparation issues.

## Data

The experiment uses the myG2P Myanmar grapheme-to-phoneme dictionary data. The source side is Myanmar grapheme syllables (`.my`), and the target side is phoneme representation (`.ph`). The cleaned split files used by the experiment are:

```text
g2p-par/train_clean.my
g2p-par/train_clean.ph
g2p-par/dev_clean.my
g2p-par/dev_clean.ph
g2p-par/test_clean.my
g2p-par/test_clean.ph
```

## Normalization and Evaluation Preparation

The dataset was normalized with `syl-Normalizer` v0.6 and `final_syl_dictionary_13Feb2024.sorted.txt`. The cleaned test split was converted to Moses SGM format using original-style segment text handling. Apostrophes in phoneme tokens, such as `te'`, are part of the target representation and must remain unchanged.

## Experiment Setup

All systems were generated from the original assignment template:

```text
pbsmt-big-normalize/config.baseline
```

The main confirmed baseline is:

```text
pbsmt_5gram_main
```

Tuning and comparison systems:

```text
pbsmt_5gram_phrase7
pbsmt_5gram_phrase11
pbsmt_5gram_phrase13
pbsmt_5gram_less_pruned
pbsmt_5gram_nbest200
pbsmt_5gram_search10000
pbsmt_3gram_phrase5
pbsmt_3gram_phrase7
pbsmt_7gram_phrase9
pbsmt_7gram_phrase11
```

## Results

{result_section}

## Comparison

{comparison_section}

The tuning results should be interpreted conservatively. Small differences of a few exact-match lines do not show that phrase length, n-gram order, pruning, n-best size, or stronger search is the real bottleneck.

## Error Analysis

{error_section}

{error_interpretation}{leaderboard_section}

## Findings

1. **PBSMT tuning does not substantially improve the model.** The tuning runs change exact-match results only marginally.

2. **The largest error group is Myanmar G2P pronunciation variation.** The previously named ambiguity category is better understood as context-dependent pronunciation behavior in Myanmar. Some words or syllables naturally have different pronunciations depending on lexical and phonological context.

3. **Data coverage and preparation still matter.** OOV and missing pronunciation variants are actionable data problems. These can be improved by dictionary expansion, data correction, and adding verified pronunciation variants.

4. **Decoder search is not the main issue.** Stronger search does not materially improve the result, so further Moses tuning is unlikely to produce large gains.


## Conclusion

The confirmed 5-gram PBSMT baseline is strong, and additional PBSMT tuning gives only marginal improvements. The main bottleneck is linguistic and data-driven: Myanmar words can have context-dependent pronunciations, and the dataset still contains OOV or missing-variant cases.

The best next step is a hybrid improvement strategy: keep the original-template PBSMT system, add a small rule-based correction layer for high-frequency pronunciation variation, and improve the dataset with targeted augmentation and data fixes.

## Generated Artifacts

```text
pbsmt_experiment_results.csv
pbsmt_error_summary_report_labels.csv
pbsmt_error_leaderboard_report_labels.csv
```

## Sources

- Ye Kyaw Thu, `myG2P`: https://github.com/ye-kyaw-thu/myG2P
- Ye Kyaw Thu, `syl-Normalizer`: https://github.com/ye-kyaw-thu/syl-Normalizer
"""

Path('README.md').write_text(readme, encoding='utf-8')
print('README.md updated')

README.md updated
